# Exercise 16: Average Selling Price

**Source:** [LeetCode 1251](https://leetcode.com/problems/average-selling-price/description/?envType=study-plan-v2&envId=top-sql-50)

Table: `Prices`
```
+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| product_id    | int     |
| start_date    | date    |
| end_date      | date    |
| price         | int     |
+---------------+---------+
```
- `(product_id, start_date, end_date)` is the primary key.
- For each `product_id` different intervals `start_date` and `end_date` are not allowed to intersect.
- `price` indicates the price during the time interval.
 

Table: `UnitsSold`
```
+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| product_id    | int     |
| purchase_date | date    |
| units         | int     |
+---------------+---------+
```
- Each row indicates number of `units` sold at the `purchase_date`.
- Duplicate rows are possible.

**Task.**
Find the average selling price `average_price` **rounded to 2 decimals** for every product.
Return 0 if a `product_id` has not been sold.
The result can be returned **any order**.

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_16 WITH (FORCE);
%sql CREATE DATABASE exercise_16;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_16 --alias exercise_16

Connecting and switching to connection 'exercise_16'

#### Fill the database with content

In [8]:
%%sql

DROP TABLE IF EXISTS Prices;

CREATE TABLE Prices (
    product_id INTEGER NOT NULL,
    start_date DATE NOT NULL,
    end_date DATE NOT NULL,
    price INTEGER NOT NULL,
    PRIMARY KEY (product_id, start_date, end_date)
);

INSERT INTO Prices (product_id, start_date, end_date, price)
VALUES
(1, '2019-02-17', '2019-02-28', 5),
(1, '2019-03-01', '2019-03-22', 20),
(2, '2019-02-01', '2019-02-20', 15),
(2, '2019-02-21', '2019-03-31', 30);

CREATE TABLE UnitsSold (
    product_id INTEGER NOT NULL,
    purchase_date DATE NOT NULL,
    units INTEGER NOT NULL
);

INSERT INTO UnitsSold (product_id, purchase_date, units)
VALUES
(1, '2019-02-25', 100),
(1, '2019-03-01', 15),
(2, '2019-02-10', 200),
(2, '2019-03-22', 30);

SELECT * FROM UnitsSold;

Running query in 'exercise_16'

4 rows affected.

4 rows affected.

4 rows affected.

product_id,purchase_date,units
1,2019-02-25,100
1,2019-03-01,15
2,2019-02-10,200
2,2019-03-22,30


#### Solve the exercise

**Task.**
Find the average selling price `average_price` **rounded to 2 decimals** for every product.
Return 0 if a `product_id` has not been sold.
The result can be returned **any order**.

In [9]:
%%sql
SELECT
    product_id, 
    CASE WHEN SUM(units) > 0 THEN ROUND(SUM(total)::NUMERIC/SUM(units),2)
         ELSE 0
    END as average_price
FROM (
    SELECT P.product_id, U.units, U.units * P.price AS total
    FROM Prices P LEFT JOIN UnitsSold U
    ON P.product_id = U.product_id
    WHERE U.purchase_date IS NULL OR U.purchase_date BETWEEN P.start_date AND P.end_date)
GROUP BY 1;

Running query in 'exercise_16'

2 rows affected.

product_id,average_price
1,6.96
2,16.96


#### Explanation

- Do a subquery which provides `P.product_id`
(it is important to take the attribute from the `Prices` table as there might be product which has not been sold, but must be listed with average price `0`), the `U.units` and the total price for these units `U.units * P.price`.
- To make sure that the `U.units` is multiplied with the correct `P.price`, ensure that the `U.purchase_date` lies between the `start_date` and `end_date`.
- In the subquery one does a `LEFT JOIN` so that all `product_id`s from `Prices` are in the table, one obtains from the subquery.
- Then define the `average_price`.
- If `SUM(units)` $= 0$, then the average price must be `0` (do this in the `CASE WHEN ... THEN ... ELSE ...END`-block).
- Else `SUM` over all `U.units * P.price` with the same product id and do a float division by the total number of units `SUM(units)`. To make sure we obtain a rational number as a result, one needs `::NUMERIC/` divison.
- Further `ROUND( _ , 2)` rounds to two units.

#### Close connection

In [10]:
%sql --close exercise_16
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_16;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)